# MarketingIQ tiny planner — Phase 6: merge, GGUF export, quantization, smoke test

**Changes the model's representation, not its behaviour.**

`unsloth/Qwen3-0.6B + Phase 4 LoRA → merged 16-bit model → GGUF (F16) → Q8_0 and Q4_K_M → llama.cpp (CPU)`

**Rules for this phase**
- **No training.** No trainer, no optimizer, no gradients. The adapter files are SHA-256 hashed before and after, and must
  equal the hashes recorded in the Phase 5 report (the exact model Phase 5 evaluated).
- `eval_frozen.jsonl` is **not** in the bundle and is not used. The smoke test uses 30 questions from `validation.jsonl`
  (`benchmark/benchmark_questions.jsonl`; 8 of them are the smoke set).
- Same prompt as Phase 4/5 (`plan_format.messages()` + Qwen3 chat template, **thinking OFF**), greedy decoding, stop at `<|im_end|>`.
- Nothing is changed because of smoke-test results: no quantization or setting is picked from them.

Step-by-step instructions: `experiments/tiny_model/PHASE6_INSTRUCTIONS.md`.

## 0. Configuration

In [ ]:
MODEL_ID = "unsloth/Qwen3-0.6B"                                    # Phase 4 base model

DRIVE_DIR         = "/content/drive/MyDrive/marketingiq_tiny"
EXPORT_BUNDLE_ZIP = f"{DRIVE_DIR}/marketingiq_export_bundle.zip"   # made by notebooks/make_export_bundle.py
RUN_NAME   = "marketingiq-qwen3-0.6b-lora"                          # Phase 4 run folder suffix
PHASE4_RUN = None   # folder under runs/, e.g. "20261001-101500_marketingiq-qwen3-0.6b-lora"; None = the only one
PHASE5_RUN = None   # folder under phase5/, e.g. "20261003-120000_frozen_eval"; None = the only one

# The Phase 5 result this export must reproduce the model of (from the Phase 5 report, used only as an identity check)
PHASE5_EXPECTED = {"exact_plan_match": (391, 429), "execution_match": (383, 429)}

WORK_DIR   = "/content/marketingiq_export"      # bundle is unzipped here (temporary)
OUTPUT_DIR = "/content/phase6_outputs"          # merged model + GGUFs (temporary, copied to Drive)
EXPORT_NAME = "marketingiq-qwen3-0.6b"
QUANTS = ["Q8_0", "Q4_K_M"]

# llama.cpp: pinned release tag (checked Oct 2026: latest release, Windows CPU zip llama-b11461-bin-win-cpu-x64.zip).
# Use the SAME tag on the laptop so conversion, quantization and inference come from one llama.cpp version.
LLAMA_CPP_TAG = "b11461"

SAVE_MERGED_TO_DRIVE = True    # merged 16-bit HF model (~1.2 GB): the source for any later re-export
SAVE_F16_GGUF_TO_DRIVE = False # unquantized GGUF (~1.2 GB); re-creatable from the merged model
SEED = 20260929

## 1. Environment Setup
In Colab: **Runtime → Change runtime type → T4 GPU** (the GPU Phase 4/5 used, so the merge happens in the same fp16
dtype Phase 5 evaluated in). The install cell is identical to Phase 4/5.

In [ ]:
import sys, subprocess, platform, os
print("Python", sys.version.split()[0], "on", platform.platform())

def stop(message):
    raise RuntimeError("\n\n" + "=" * 70 + "\nSTOPPED: " + message + "\n" + "=" * 70)

import torch
if not torch.cuda.is_available():
    stop("No CUDA GPU. In Colab: Runtime -> Change runtime type -> T4 GPU, then Runtime -> Restart and run again.")
props = torch.cuda.get_device_properties(0)
GPU_NAME, GPU_VRAM_GB = props.name, round(props.total_memory / 1024**3, 1)
print(f"GPU: {GPU_NAME} | VRAM: {GPU_VRAM_GB} GB | CPU cores: {os.cpu_count()}")

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2
!pip install duckdb psutil

In [ ]:
import unsloth  # import before transformers so Unsloth's patches apply
import importlib.metadata as md_
VERSIONS = {p: md_.version(p) for p in ["unsloth", "unsloth_zoo", "torch", "transformers", "peft", "accelerate",
                                        "safetensors", "duckdb"]}
VERSIONS["python"] = sys.version.split()[0]
for k, v in VERSIONS.items():
    print(f"{k:14s} {v}")

## 2. Verify the Phase 4 adapter = the model Phase 5 evaluated
Finds the Phase 4 run and the Phase 5 report in Drive. The adapter's files must exist and their SHA-256 must equal the
hashes recorded in `frozen_eval_report.json`. Any mismatch stops here: no export of the wrong adapter.

In [ ]:
import json, hashlib, shutil, zipfile, datetime, time
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive")

def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for block in iter(lambda: fh.read(chunk), b""):
            h.update(block)
    return h.hexdigest()

def hash_dir(d):
    return {str(p.relative_to(d)): sha256_file(p) for p in sorted(Path(d).rglob("*")) if p.is_file()}

def only_one(folder, pattern, chosen, what):
    if chosen:
        return folder / chosen
    found = sorted(folder.glob(pattern))
    if len(found) != 1:
        stop(f"{len(found)} {what} found in {folder}: {[p.name for p in found]}. Set it in section 0.")
    return found[0]

RUN_DIR = only_one(Path(DRIVE_DIR) / "runs", f"*_{RUN_NAME}", PHASE4_RUN, "Phase 4 runs")
ADAPTER_DIR = RUN_DIR / "adapter"
P5_DIR = only_one(Path(DRIVE_DIR) / "phase5", "*_frozen_eval", PHASE5_RUN, "Phase 5 results")

REQUIRED = ["adapter_config.json", "adapter_model.safetensors", "marketingiq_prompt_contract.json",
            "tokenizer.json", "tokenizer_config.json"]
missing = [f for f in REQUIRED if not (ADAPTER_DIR / f).is_file()]
for f in ("training_report.json",):
    if not (RUN_DIR / f).is_file(): missing.append(str(RUN_DIR / f))
if not (P5_DIR / "frozen_eval_report.json").is_file(): missing.append(str(P5_DIR / "frozen_eval_report.json"))
if missing:
    stop(f"Missing files: {missing}")

TRAIN_REPORT = json.loads((RUN_DIR / "training_report.json").read_text())
REPORT5 = json.loads((P5_DIR / "frozen_eval_report.json").read_text())
CONTRACT = json.loads((ADAPTER_DIR / "marketingiq_prompt_contract.json").read_text())
ADAPTER_CONFIG = json.loads((ADAPTER_DIR / "adapter_config.json").read_text())
ADAPTER_SHA_BEFORE = hash_dir(ADAPTER_DIR)

checks = {
    "Phase 4 report is phase 4": TRAIN_REPORT.get("phase") == 4,
    "Phase 4 model_id is the base model": TRAIN_REPORT.get("model_id") == MODEL_ID,
    "Phase 4 trained with thinking OFF": TRAIN_REPORT.get("thinking_mode") == "OFF",
    "Phase 4 did not use the frozen set": TRAIN_REPORT.get("frozen_eval_used") is False,
    "adapter base model is the base model": str(ADAPTER_CONFIG.get("base_model_name_or_path", "")).lower() == MODEL_ID.lower(),
    "adapter prompt contract == training report": CONTRACT == TRAIN_REPORT.get("prompt_contract"),
    "Phase 5 report is phase 5, no training": REPORT5.get("phase") == 5 and REPORT5.get("training_performed") is False,
    "Phase 5 evaluated this Phase 4 run": REPORT5["model"]["phase4_run"] == RUN_DIR.name,
    "adapter SHA-256 == Phase 5 report (every file)": ADAPTER_SHA_BEFORE == REPORT5["model"]["adapter_sha256"],
    "Phase 5 adapter was unmerged/unquantized": not REPORT5["model"]["merged"] and not REPORT5["model"]["quantized"],
}
m5 = REPORT5["metrics"]
for key, (k, n) in PHASE5_EXPECTED.items():
    checks[f"Phase 5 {key} == {k}/{n}"] = (m5[key]["correct"], m5[key]["total"]) == (k, n)
for name, ok in checks.items():
    print(f"  {'OK  ' if ok else 'FAIL'}  {name}")
if not all(checks.values()):
    if ADAPTER_SHA_BEFORE != REPORT5["model"]["adapter_sha256"]:
        diff = {k for k in set(ADAPTER_SHA_BEFORE) | set(REPORT5["model"]["adapter_sha256"])
                if ADAPTER_SHA_BEFORE.get(k) != REPORT5["model"]["adapter_sha256"].get(k)}
        print("  files that differ from the Phase 5 record:", sorted(diff))
    stop("ADAPTER VERIFICATION: FAIL - this is not the adapter Phase 5 evaluated. Nothing exported.")

MAX_SEQ_LENGTH, MAX_NEW_TOKENS = CONTRACT["max_seq_length"], CONTRACT["max_new_tokens"]
print("\nADAPTER VERIFICATION: PASS")
print("Phase 4 run:   ", RUN_DIR.name, "| Phase 5 run:", P5_DIR.name)
print("LoRA:          ", f"r={ADAPTER_CONFIG.get('r')} alpha={ADAPTER_CONFIG.get('lora_alpha')} "
      f"dropout={ADAPTER_CONFIG.get('lora_dropout')} targets={sorted(ADAPTER_CONFIG.get('target_modules') or [])}")
print("Contract:      ", json.dumps(CONTRACT))
print("Adapter SHA-256:", {k: v[:12] for k, v in ADAPTER_SHA_BEFORE.items()})
print("Phase 5 dtype / GPU:", REPORT5["model"]["dtype"], "/", REPORT5["environment"]["gpu"], "| this session:", GPU_NAME)

## 3. Export bundle
The Phase 4 plan format, Phase 2 scorer, Phase 5 harness functions, backend + CSV (hash-checked by
`make_export_bundle.py`), the 30 benchmark questions and the llama-server client.

In [ ]:
if not os.path.isfile(EXPORT_BUNDLE_ZIP):
    stop(f"Bundle not found at {EXPORT_BUNDLE_ZIP}. Upload marketingiq_export_bundle.zip to MyDrive/marketingiq_tiny/.")
shutil.rmtree(WORK_DIR, ignore_errors=True)
with zipfile.ZipFile(EXPORT_BUNDLE_ZIP) as z:
    names = z.namelist()
    if any(s in n for n in names for s in ("eval_frozen", "train.jsonl", "validation.jsonl")):
        stop("The export bundle contains evaluation or training data; rebuild it with make_export_bundle.py.")
    z.extractall(WORK_DIR)
BUNDLE_MANIFEST = json.loads((Path(WORK_DIR) / "bundle_manifest.json").read_text())
bad = [rel for rel, h in BUNDLE_MANIFEST.items() if sha256_file(Path(WORK_DIR) / rel) != h]
if bad:
    stop(f"Unzipped files differ from the bundle manifest: {bad}")
P5_CODE = REPORT5.get("code_sha256", {})
drift = [rel for rel, h in BUNDLE_MANIFEST.items() if rel in P5_CODE and P5_CODE[rel] != h]
if drift:
    stop(f"Code differs from what Phase 5 evaluated with: {drift}")

TINY = Path(WORK_DIR) / "experiments" / "tiny_model"
sys.path[:0] = [str(TINY / "evaluation"), str(TINY / "benchmark")]
import frozen_eval as fe      # Phase 5 harness: build_prompt, generate, check_lora_loaded, lora_digest, score_example
import llama_client as lc     # Phase 6 llama-server client + exact prompt renderer
pf = fe.pf
pf.backend()
if CONTRACT["system_prompt"] != pf.SYSTEM_PROMPT or CONTRACT["chat_template_kwargs"] != {"enable_thinking": False} \
        or CONTRACT["stop_token"] != fe.EOS:
    stop("Prompt contract differs from the bundled plan_format.py / harness.")

ROWS = lc.load_questions(TINY / "benchmark" / "benchmark_questions.jsonl")
SMOKE_IDS = [r["id"] for r in ROWS if r["smoke"]]
gold_self = [fe.score_example(r, {"raw_output": json.dumps(r["plan"], separators=(",", ":")), "finished": True,
                                  "new_tokens": 0, "raw_with_special": json.dumps(r["plan"]) + fe.EOS}) for r in ROWS]
if not all(s["exact"] and s["validate_plan_ok"] for s in gold_self):
    stop("Scoring self-test failed on the benchmark questions.")
print(f"Bundle OK ({len(names)} files, identical to the Phase 5 code where shared). "
      f"{len(ROWS)} benchmark questions, smoke set: {SMOKE_IDS}. Gold self-test: all exact.")

## 4. Load Qwen3-0.6B + the Phase 4 LoRA exactly as Phase 5 (inference only)
Same call as Phase 5 (adapter folder, 16-bit, `max_seq_length` from the contract). The live LoRA tensors must equal
`adapter_model.safetensors`, and the LoRA fingerprint is compared with the one Phase 5 recorded.

In [ ]:
from unsloth import FastLanguageModel
from transformers import AutoTokenizer, set_seed
set_seed(SEED)
torch.set_grad_enabled(False)          # no gradients anywhere in this notebook

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = str(ADAPTER_DIR),
    max_seq_length = MAX_SEQ_LENGTH,
    dtype = None,
    load_in_4bit = False,
    load_in_8bit = False,
    full_finetuning = False,
)
FastLanguageModel.for_inference(model)
model.eval()

if str(model.peft_config["default"].base_model_name_or_path).lower() != MODEL_ID.lower():
    stop("Loaded base model is not " + MODEL_ID)
if tokenizer.eos_token != fe.EOS:
    stop(f"eos token {tokenizer.eos_token!r} != {fe.EOS!r}")
try:
    LORA_INFO = fe.check_lora_loaded(model, ADAPTER_DIR)
except fe.IntegrityError as e:
    stop(str(e))
LORA_DIGEST = fe.lora_digest(model)
same_dtype = str(model.dtype) == REPORT5["model"]["dtype"]
if same_dtype and LORA_DIGEST != REPORT5["model"]["lora_weights_digest"]:
    stop("Live LoRA weights differ from the fingerprint Phase 5 recorded.")
print(f"dtype {model.dtype} | LoRA tensors {LORA_INFO['lora_tensors']} ({LORA_INFO['lora_parameters']:,} params) == saved adapter")
print("LoRA fingerprint == Phase 5:", LORA_DIGEST == REPORT5["model"]["lora_weights_digest"],
      "" if same_dtype else f"(not comparable: dtype {model.dtype} vs Phase 5 {REPORT5['model']['dtype']})")
print(f"eos {tokenizer.eos_token!r} (id {tokenizer.convert_tokens_to_ids(fe.EOS)}) | bos {tokenizer.bos_token!r} | "
      f"pad {tokenizer.pad_token!r} | max_seq_length {MAX_SEQ_LENGTH} | max_new_tokens {MAX_NEW_TOKENS}")

# Prompts: Phase 5's builder (HF chat template, thinking OFF) == the hand-rendered string llama.cpp will receive.
PROMPTS = [fe.build_prompt(tokenizer, r) for r in ROWS]
hub_tok = AutoTokenizer.from_pretrained(MODEL_ID)
diff = [r["id"] for r, p in zip(ROWS, PROMPTS)
        if p != lc.render_prompt(r["question"], r["previous"]) or p != fe.build_prompt(hub_tok, r)]
if diff:
    stop(f"Prompt rendering differs for {diff}")
PROMPT_IDS = [tokenizer(p, add_special_tokens=False)["input_ids"] for p in PROMPTS]
print(f"\n{len(PROMPTS)} prompts: HF template (thinking OFF) == llama_client.render_prompt() == Hub template")
print(PROMPTS[next(i for i, r in enumerate(ROWS) if r["previous"])])

## 5. Reference generations (LoRA, unmerged, Phase 5 settings)
Greedy, batch size 1 (no padding, like llama.cpp serving one request). These outputs are the reference the merged
model and every GGUF are compared with. 30 validation questions; never the frozen set.

In [ ]:
t0 = time.time()
REF = fe.generate(model, tokenizer, PROMPTS, MAX_NEW_TOKENS, 1)
REF_SCORES = [fe.score_example(r, g) for r, g in zip(ROWS, REF)]
print(f"{len(REF)} reference plans in {time.time() - t0:.0f}s")
for k in ("finished", "parsed", "validate_plan_ok", "exact", "execution_match"):
    print(f"  {k:18s} {sum(bool(s[k]) for s in REF_SCORES)}/{len(REF)}")
print("  thinking in output", sum(s["thinking_in_output"] for s in REF_SCORES))

## 6. Merge LoRA into the base weights (official Unsloth merge)
`save_pretrained_merged(..., save_method="merged_16bit")` writes base + LoRA deltas as one 16-bit Hugging Face model
(the Unsloth-documented step before llama.cpp conversion). The adapter folder is never written to.

Two independent checks:
1. **Weights:** for every LoRA-targeted matrix, merged `W` must equal `W_base + (alpha/r)·B·A` (computed here in float32
   from the Hub base weights and the adapter file) up to 16-bit rounding; every other tensor must equal the base.
2. **Behaviour:** the merged model, loaded on its own (no adapter, no PEFT modules), must generate the same plans as section 5.

In [ ]:
OUT = Path(OUTPUT_DIR)
MERGED_DIR = OUT / "merged_16bit"
shutil.rmtree(OUT, ignore_errors=True)
OUT.mkdir(parents=True)

model.save_pretrained_merged(str(MERGED_DIR), tokenizer, save_method="merged_16bit")

if hash_dir(ADAPTER_DIR) != ADAPTER_SHA_BEFORE:
    stop("Adapter files changed during the merge.")
if fe.lora_digest(model) != LORA_DIGEST:
    stop("In-memory LoRA weights changed during the merge.")
MERGED_FILES = sorted(p.name for p in MERGED_DIR.iterdir())
print("merged files:", MERGED_FILES)
if any(n.startswith("adapter_") for n in MERGED_FILES) or not any(n.endswith(".safetensors") for n in MERGED_FILES):
    stop("Merged folder is not a plain (merged) model.")
MERGED_CONFIG = json.loads((MERGED_DIR / "config.json").read_text())
MERGED_DTYPE = MERGED_CONFIG.get("torch_dtype") or MERGED_CONFIG.get("dtype")
print("architecture:", MERGED_CONFIG.get("architectures"), "| dtype:", MERGED_DTYPE,
      "| tie_word_embeddings:", MERGED_CONFIG.get("tie_word_embeddings"))

In [ ]:
# Check 1: weights. Base (Hub) + (alpha/r)·B·A in float32 vs the merged tensors.
from safetensors import safe_open
from safetensors.torch import load_file
from huggingface_hub import snapshot_download

def read_tensors(folder):
    out = {}
    for f in sorted(Path(folder).glob("*.safetensors")):
        with safe_open(str(f), "pt") as st:
            for k in st.keys():
                out[k] = st.get_tensor(k)
    return out

base_dir = snapshot_download(MODEL_ID, allow_patterns=["*.safetensors", "*.json"])
BASE_W = read_tensors(base_dir)
MERGED_W = read_tensors(MERGED_DIR)
LORA_W = load_file(str(ADAPTER_DIR / "adapter_model.safetensors"))
scale = ADAPTER_CONFIG["lora_alpha"] / ADAPTER_CONFIG["r"]
if ADAPTER_CONFIG.get("use_rslora"):
    stop("rsLoRA scaling is not expected for this adapter.")

def lora_pair(weight_key):
    stem = weight_key[:-len(".weight")]
    a = [k for k in LORA_W if k.endswith(stem + ".lora_A.weight")]
    b = [k for k in LORA_W if k.endswith(stem + ".lora_B.weight")]
    return (LORA_W[a[0]], LORA_W[b[0]]) if len(a) == len(b) == 1 else None

if set(BASE_W) - set(MERGED_W) - {"lm_head.weight"}:
    stop(f"Merged model is missing tensors: {sorted(set(BASE_W) - set(MERGED_W))[:5]}")
merged_n, other_n, worst_merged, worst_other = 0, 0, 0.0, 0.0
for k, w0 in BASE_W.items():
    if k not in MERGED_W:
        continue
    w = MERGED_W[k].float()
    pair = lora_pair(k)
    if pair is not None:
        A, B = pair
        expected = w0.float() + scale * (B.float() @ A.float())
        rel = ((w - expected).abs().max() / expected.abs().max()).item()
        worst_merged, merged_n = max(worst_merged, rel), merged_n + 1
    else:
        rel = ((w - w0.float()).abs().max() / w0.float().abs().max().clamp(min=1e-12)).item()
        worst_other, other_n = max(worst_other, rel), other_n + 1
n_lora_targets = sum(1 for k in LORA_W if k.endswith("lora_A.weight"))
print(f"LoRA-merged matrices checked: {merged_n} (adapter has {n_lora_targets}) | worst relative diff {worst_merged:.2e}")
print(f"Other tensors checked: {other_n} | worst relative diff vs base {worst_other:.2e}")
WEIGHT_CHECK = {"lora_matrices": merged_n, "adapter_lora_pairs": n_lora_targets, "scale": scale,
                "max_rel_diff_merged": worst_merged, "max_rel_diff_other": worst_other}
if merged_n != n_lora_targets or worst_merged > 1e-2 or worst_other > 1e-2:
    stop("WEIGHT CHECK: FAIL - merged weights are not base + (alpha/r)·B·A.")
print("WEIGHT CHECK: PASS (differences are 16-bit rounding only)")
del BASE_W, MERGED_W

In [ ]:
# Check 2: behaviour. The merged model alone (no adapter, no PEFT) vs the LoRA reference.
# Loaded with Unsloth's loader: importing unsloth patches the Qwen3 classes process-wide, and a plain
# AutoModelForCausalLM then fails in generate() ('max_seq_length' missing). The merged folder has no adapter
# files, so this loads only the merged weights - and both sides run the same (patched) Qwen3 code.
merged_tok = AutoTokenizer.from_pretrained(str(MERGED_DIR))
if [fe.build_prompt(merged_tok, r) for r in ROWS] != PROMPTS:
    stop("The merged model's tokenizer/chat template renders different prompts.")
if merged_tok.eos_token != fe.EOS:
    stop(f"Merged tokenizer eos {merged_tok.eos_token!r}")
merged_model, _ = FastLanguageModel.from_pretrained(
    model_name = str(MERGED_DIR),
    max_seq_length = MAX_SEQ_LENGTH,
    dtype = model.dtype,
    load_in_4bit = False,
    load_in_8bit = False,
    full_finetuning = False,
)
FastLanguageModel.for_inference(merged_model)
merged_model.eval()
if hasattr(merged_model, "peft_config") or any("lora_" in n for n, _ in merged_model.named_parameters()):
    stop("The merged model was loaded with LoRA/PEFT modules; it must be the plain merged weights.")
print(f"merged model loaded: {type(merged_model).__name__}, dtype {merged_model.dtype}, no LoRA modules")
MERGED_GEN = fe.generate(merged_model, merged_tok, PROMPTS, MAX_NEW_TOKENS, 1)
MERGE_DIFF = [r["id"] for r, a, b in zip(ROWS, REF, MERGED_GEN) if a["raw_output"] != b["raw_output"]]
MERGED_SCORES = [fe.score_example(r, g) for r, g in zip(ROWS, MERGED_GEN)]
print(f"Merged vs LoRA reference: {len(ROWS) - len(MERGE_DIFF)}/{len(ROWS)} identical outputs; differ: {MERGE_DIFF}")
for i in [ROWS.index(next(r for r in ROWS if r["id"] == d)) for d in MERGE_DIFF]:
    print(f"  {ROWS[i]['id']}: LoRA {REF[i]['raw_output']}\n  {'':10s} merged {MERGED_GEN[i]['raw_output']}")
print("merged validate_plan:", sum(s["validate_plan_ok"] for s in MERGED_SCORES), "/", len(ROWS))
del merged_model; torch.cuda.empty_cache()

## 7. GGUF conversion and quantization (llama.cpp, pinned tag)
Upstream llama.cpp tools at the pinned release tag (the same tools Unsloth's `save_pretrained_gguf` runs):
- `convert_hf_to_gguf.py` → unquantized GGUF in the merged model's own 16-bit type (lossless), with the tokenizer,
  special tokens, EOS and the Qwen3 chat template embedded;
- `llama-quantize` → `Q8_0` and `Q4_K_M`;
- `llama-server` (CPU build, `GGML_CUDA=OFF`) for the smoke test.

The CPU-only build takes ~5–10 minutes on Colab.

In [ ]:
LLAMA = Path("/content/llama.cpp")
def run(cmd, log=None, **kw):
    print("$", " ".join(map(str, cmd)))
    p = subprocess.run(list(map(str, cmd)), capture_output=True, text=True, **kw)
    if log:
        Path(log).write_text(p.stdout + "\n" + p.stderr)
    if p.returncode != 0:
        print(p.stdout[-3000:], p.stderr[-3000:])
        stop(f"command failed ({p.returncode}): {cmd[0]} ...")
    return p

shutil.rmtree(LLAMA, ignore_errors=True)
run(["git", "clone", "--depth", "1", "--branch", LLAMA_CPP_TAG, "https://github.com/ggml-org/llama.cpp", LLAMA])
LLAMA_COMMIT = run(["git", "-C", LLAMA, "rev-parse", "HEAD"]).stdout.strip()
print("llama.cpp", LLAMA_CPP_TAG, "commit", LLAMA_COMMIT)
run([sys.executable, "-m", "pip", "install", "-q", str(LLAMA / "gguf-py")])
run(["cmake", "-S", LLAMA, "-B", LLAMA / "build", "-DGGML_CUDA=OFF", "-DBUILD_SHARED_LIBS=OFF",
     "-DLLAMA_BUILD_TESTS=OFF", "-DCMAKE_BUILD_TYPE=Release"], log=OUT / "cmake_configure.log")
t0 = time.time()
run(["cmake", "--build", LLAMA / "build", "--config", "Release", "-j", str(os.cpu_count()),
     "--target", "llama-quantize", "llama-server"], log=OUT / "cmake_build.log")
BIN = LLAMA / "build" / "bin"
print(f"built in {time.time() - t0:.0f}s:", sorted(p.name for p in BIN.iterdir()))

In [ ]:
EXPORT = OUT / "export"
EXPORT.mkdir(exist_ok=True)
OUTTYPE = {"float16": "f16", "bfloat16": "bf16"}.get(str(MERGED_DTYPE).replace("torch.", ""))
if OUTTYPE is None:
    stop(f"Unexpected merged dtype {MERGED_DTYPE}")
GGUF = {OUTTYPE.upper(): EXPORT / f"{EXPORT_NAME}-{OUTTYPE.upper()}.gguf"}
COMMANDS = {}
cmd = [sys.executable, LLAMA / "convert_hf_to_gguf.py", MERGED_DIR, "--outtype", OUTTYPE, "--outfile", GGUF[OUTTYPE.upper()]]
p = subprocess.run(list(map(str, cmd)), capture_output=True, text=True)
(OUT / "convert.log").write_text(p.stdout + "\n" + p.stderr)
if p.returncode != 0:
    print(p.stderr[-4000:])
    stop("convert_hf_to_gguf.py failed. If it is a missing Python package, run\n"
         f"  !pip install -r {LLAMA}/requirements/requirements-convert_hf_to_gguf.txt\nand re-run this cell.")
COMMANDS[OUTTYPE.upper()] = " ".join(map(str, cmd)).replace(str(OUT) + "/", "")
for q in QUANTS:
    GGUF[q] = EXPORT / f"{EXPORT_NAME}-{q}.gguf"
    cmd = [BIN / "llama-quantize", GGUF[OUTTYPE.upper()], GGUF[q], q]
    run(cmd, log=OUT / f"quantize_{q}.log")
    COMMANDS[q] = " ".join(map(str, cmd)).replace(str(OUT) + "/", "").replace(str(BIN) + "/", "")
GGUF_INFO = {}
for q, path in GGUF.items():
    GGUF_INFO[q] = {"file": path.name, "size_bytes": path.stat().st_size,
                    "size_mb": round(path.stat().st_size / 2**20, 1), "sha256": sha256_file(path), "command": COMMANDS[q]}
    print(f"{q:7s} {GGUF_INFO[q]['size_mb']:8.1f} MB  {GGUF_INFO[q]['sha256']}  {path.name}")

In [ ]:
# GGUF metadata: architecture, tokenizer, special tokens, chat template.
from gguf import GGUFReader

def gguf_meta(path):
    r = GGUFReader(str(path))
    def val(key):
        f = r.fields.get(key)
        if f is None:
            return None
        if hasattr(f, "contents"):
            return f.contents()
        part = f.parts[f.data[0]]
        return bytes(part).decode() if f.types and f.types[0].name == "STRING" else part.tolist()[0]
    tmpl = val("tokenizer.chat_template") or ""
    return {"architecture": val("general.architecture"), "file_type": val("general.file_type"),
            "block_count": val("qwen3.block_count"), "context_length": val("qwen3.context_length"),
            "tokenizer_model": val("tokenizer.ggml.model"), "tokenizer_pre": val("tokenizer.ggml.pre"),
            "eos_token_id": val("tokenizer.ggml.eos_token_id"), "bos_token_id": val("tokenizer.ggml.bos_token_id"),
            "padding_token_id": val("tokenizer.ggml.padding_token_id"), "add_bos_token": val("tokenizer.ggml.add_bos_token"),
            "chat_template_present": bool(tmpl), "chat_template_supports_enable_thinking": "enable_thinking" in tmpl,
            "chat_template_sha256": hashlib.sha256(tmpl.encode()).hexdigest() if tmpl else None,
            "tensors": len(r.tensors)}

EOS_ID = tokenizer.convert_tokens_to_ids(fe.EOS)
GGUF_META = {q: gguf_meta(p) for q, p in GGUF.items()}
for q, m in GGUF_META.items():
    print(q, json.dumps(m))
    problems = [k for k, ok in {"architecture qwen3": m["architecture"] == "qwen3",
                                f"eos id {EOS_ID} (<|im_end|>)": m["eos_token_id"] == EOS_ID,
                                "no BOS added": not m["add_bos_token"],
                                "chat template embedded": m["chat_template_present"]}.items() if not ok]
    if problems:
        stop(f"{q}: GGUF metadata check failed: {problems}")
print("GGUF METADATA: PASS (qwen3, <|im_end|> EOS, no BOS, chat template embedded)")

## 8. ▶ Smoke test through llama.cpp (CPU)
For each GGUF a fresh `llama-server` (CPU, the same client the laptop benchmark uses):
1. llama.cpp must tokenize every prompt to **exactly** the Hugging Face token ids (special tokens included, no BOS);
2. the GGUF's embedded template, rendered by llama.cpp with `enable_thinking=false`, is compared with the prompt (recorded);
3. greedy generation on the 30 benchmark questions → parse → `to_canonical()` → `validate_plan()` → execution.

**Smoke PASS** for a GGUF = on the 8 smoke questions every output stops at EOS, contains no thinking, parses,
canonicalizes and passes `validate_plan()`. Agreement with the section 5 reference is recorded for all 30 (a quantized
model on CPU may differ in a few outputs; nothing is tuned on it). Colab CPU timings are not the laptop benchmark.

In [ ]:
SMOKE, SMOKE_ROWS = {}, []
for q, path in GGUF.items():
    with lc.LlamaServer(BIN / "llama-server", path, threads=os.cpu_count(), ctx=512, port=8089,
                        log_path=OUT / f"server_{q}.log") as srv:
        props = srv.props()
        tok_diff = [r["id"] for r, p, ids in zip(ROWS, PROMPTS, PROMPT_IDS) if srv.tokenize(p) != ids]
        try:
            tmpl_same = [srv.apply_template(pf.messages(r["question"], r["previous"]), enable_thinking=False) == p
                         for r, p in zip(ROWS, PROMPTS)]
            template_check = f"{sum(tmpl_same)}/{len(ROWS)} identical"
        except Exception as e:  # informational only: inference uses the raw prompt
            template_check = f"not checked ({type(e).__name__}: {e})"
        recs = []
        for r, p, ids, ref in zip(ROWS, PROMPTS, PROMPT_IDS, REF):
            res = srv.complete(p, MAX_NEW_TOKENS, cache_prompt=False)
            s = fe.score_example(r, lc.as_generation(res))
            recs.append({"gguf": q, "id": r["id"], "category": r["category"], "smoke": r["smoke"],
                         "question": r["question"], "previous": r["previous"], "raw_output": res["text"],
                         "stop_type": res["stop_type"], "prompt_tokens_evaluated": res["tokens_evaluated"],
                         "prompt_tokens_hf": len(ids), "same_as_reference": res["text"].strip() == ref["raw_output"],
                         **{k: s[k] for k in ("finished", "thinking_in_output", "parsed", "canonicalized",
                                              "validate_plan_ok", "validator_error", "exact", "execution_match", "outcome")}})
    smoke = [x for x in recs if x["smoke"]]
    count = lambda rows, k: sum(bool(x[k]) for x in rows)
    passed = (not tok_diff and all(x["finished"] and not x["thinking_in_output"] and x["parsed"] and x["canonicalized"]
                                   and x["validate_plan_ok"] for x in smoke)
              and all(x["prompt_tokens_evaluated"] == x["prompt_tokens_hf"] for x in recs))
    SMOKE[q] = {"pass": passed, "server_eos_token": props.get("eos_token"), "build_info": props.get("build_info"),
                "tokenization_identical": f"{len(ROWS) - len(tok_diff)}/{len(ROWS)}", "tokenization_diff_ids": tok_diff,
                "embedded_template_thinking_off": template_check,
                "smoke": {k: f"{count(smoke, k)}/{len(smoke)}" for k in
                          ("finished", "parsed", "canonicalized", "validate_plan_ok", "thinking_in_output", "exact")},
                "all30": {k: f"{count(recs, k)}/{len(recs)}" for k in
                          ("finished", "parsed", "canonicalized", "validate_plan_ok", "thinking_in_output", "exact",
                           "execution_match", "same_as_reference")}}
    SMOKE_ROWS += recs
    print(f"\n=== {q}: SMOKE {'PASS' if passed else 'FAIL'} ===")
    print(json.dumps(SMOKE[q], indent=1))
    for x in smoke:
        print(f"  {x['id']:10s} {x['category']:13s} {x['outcome']:22s} {x['raw_output']}")
if not all(v["pass"] for v in SMOKE.values()):
    print("\nWARNING: a GGUF failed the smoke test - see above. Diagnose the export; do not change the model.")

## 9. Save to Google Drive
`MyDrive/marketingiq_tiny/phase6/<date-time>_export/`: the quantized GGUFs, `marketingiq_runtime.json` (what the laptop
benchmark reads: prompt contract, n_predict, GGUF hashes), `phase6_export_report.json`, smoke results, reference
generations and logs; plus the merged 16-bit model if `SAVE_MERGED_TO_DRIVE`.

In [ ]:
if hash_dir(ADAPTER_DIR) != ADAPTER_SHA_BEFORE:
    stop("Adapter files changed during Phase 6.")
keep = [q for q in GGUF if q in QUANTS or SAVE_F16_GGUF_TO_DRIVE]
RUNTIME = {
    "model": f"{MODEL_ID} + MarketingIQ LoRA ({RUN_DIR.name}), merged", "system_prompt": pf.SYSTEM_PROMPT,
    "prompt_format": "Qwen3 ChatML, thinking OFF: llama_client.render_prompt() == apply_chat_template(..., enable_thinking=False)",
    "max_new_tokens": MAX_NEW_TOKENS, "max_seq_length": MAX_SEQ_LENGTH, "eos_token": fe.EOS, "eos_token_id": EOS_ID,
    "add_bos": False, "decoding": {"temperature": 0, "top_k": 1},
    "llama_cpp": {"tag": LLAMA_CPP_TAG, "commit": LLAMA_COMMIT},
    "gguf": {GGUF_INFO[q]["file"]: {k: GGUF_INFO[q][k] for k in ("sha256", "size_bytes")} | {"type": q} for q in keep},
}
REPORT6 = {
    "phase": 6, "created_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
    "training_performed": False, "frozen_eval_used": False, "gradients": False,
    "base_model": MODEL_ID, "phase4_run": RUN_DIR.name, "phase5_run": P5_DIR.name,
    "adapter_verified": True, "adapter_sha256": ADAPTER_SHA_BEFORE, "adapter_unchanged": True,
    "lora_config": {k: ADAPTER_CONFIG.get(k) for k in ("r", "lora_alpha", "lora_dropout", "target_modules", "use_rslora")},
    "lora_weights_digest": LORA_DIGEST, "lora_digest_matches_phase5": LORA_DIGEST == REPORT5["model"]["lora_weights_digest"],
    "prompt_contract": CONTRACT,
    "tokenizer": {"eos": tokenizer.eos_token, "eos_id": EOS_ID, "bos": tokenizer.bos_token, "pad": tokenizer.pad_token,
                  "add_special_tokens": False},
    "merge": {"method": 'Unsloth save_pretrained_merged(save_method="merged_16bit")', "dtype": str(MERGED_DTYPE),
              "files": MERGED_FILES, "weight_check": WEIGHT_CHECK,
              "merged_vs_lora_identical_outputs": f"{len(ROWS) - len(MERGE_DIFF)}/{len(ROWS)}", "differing_ids": MERGE_DIFF},
    "gguf": {q: {**GGUF_INFO[q], "metadata": GGUF_META[q], "saved_to_drive": q in keep} for q in GGUF},
    "llama_cpp": {"tag": LLAMA_CPP_TAG, "commit": LLAMA_COMMIT, "build": "CPU (GGML_CUDA=OFF)"},
    "smoke_test": SMOKE, "smoke_question_ids": SMOKE_IDS,
    "reference_lora": {k: f"{sum(bool(s[k]) for s in REF_SCORES)}/{len(ROWS)}"
                       for k in ("finished", "parsed", "validate_plan_ok", "exact", "execution_match")},
    "benchmark_questions": "benchmark/benchmark_questions.jsonl (validation.jsonl sample, 0 frozen)",
    "environment": {"gpu": GPU_NAME, "versions": VERSIONS}, "code_sha256": BUNDLE_MANIFEST, "groq_calls": 0,
}
(EXPORT / "marketingiq_runtime.json").write_text(json.dumps(RUNTIME, indent=2))
(OUT / "phase6_export_report.json").write_text(json.dumps(REPORT6, indent=2))
for name, rows in (("smoke_results.jsonl", SMOKE_ROWS), ("reference_generations.jsonl",
                   [{"id": r["id"], **g} for r, g in zip(ROWS, REF)])):
    with (OUT / name).open("w", encoding="utf-8") as fh:
        for x in rows:
            fh.write(json.dumps(x) + "\n")

stamp = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%d-%H%M%S")
DRIVE_OUT = Path(DRIVE_DIR) / "phase6" / f"{stamp}_export"
(DRIVE_OUT / "export").mkdir(parents=True)
to_copy = [EXPORT / "marketingiq_runtime.json"] + [GGUF[q] for q in keep] + \
          [p for p in OUT.iterdir() if p.is_file()]
for p in to_copy:
    shutil.copy2(p, DRIVE_OUT / p.relative_to(OUT))
if SAVE_MERGED_TO_DRIVE:
    shutil.copytree(MERGED_DIR, DRIVE_OUT / "merged_16bit")
    to_copy += [p for p in MERGED_DIR.rglob("*") if p.is_file()]
bad = [p for p in to_copy if sha256_file(p) != sha256_file(DRIVE_OUT / p.relative_to(OUT))]
if bad:
    stop(f"Copy to Drive failed for {bad}")
print("Saved to Google Drive:", DRIVE_OUT, f"({len(to_copy)} files, SHA-256 verified)")
for p in sorted(DRIVE_OUT.rglob("*")):
    if p.is_file() and p.stat().st_size > 1e6:
        print(f"{p.stat().st_size / 2**20:9.1f} MB  {p.relative_to(DRIVE_OUT)}")
print("\nPHASE 6 EXPORT:", "PASS" if all(v["pass"] for v in SMOKE.values()) else "SMOKE FAILURES - see section 8")
for q in keep:
    print(f"  {q:7s} {GGUF_INFO[q]['file']}  {GGUF_INFO[q]['size_mb']} MB  sha256 {GGUF_INFO[q]['sha256']}  "
          f"smoke {'PASS' if SMOKE[q]['pass'] else 'FAIL'}")

## Phase 6 export ends here
Download `export/` from the Drive folder into `experiments/tiny_model/export/` on the laptop (the GGUFs and
`marketingiq_runtime.json`; git-ignored) and follow `PHASE6_INSTRUCTIONS.md` for the local CPU benchmark.
Not in this phase: FastAPI integration, removing the Query Router, any production change.